# From diagnosis to a damping hypothesis

A diagnostic is operationally useful only when it selects a prospective intervention. This notebook evaluates a preregistered scalar damping grid against a previously measured reduced operator.

## Planning contract

The planner minimizes a composite CPS risk over the same selected couplings. Its recommendation is a hypothesis for a matched continuation run. It is not applied to model weights here, and it should not be described as a causal result.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

# Editable installs write a .pth file, but the running Colab kernel does not
# automatically reprocess newly-created .pth files. Put the source tree on
# sys.path explicitly so the very next cell can import CPS without a restart.
import importlib
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)


In [ ]:
from cps.notebook import show_environment
runtime = show_environment()

## Stage 1 — import and inspect a prior evidence packet

Colab notebooks generally run in separate ephemeral virtual machines. Completing notebook 04 does not place its files in notebook 05 automatically. This stage first searches the current runtime, `CPS_EVIDENCE_PATH`, the repository, and common mounted-Drive locations. When nothing is found, it opens a browser chooser: upload the `cps-export.zip` produced by notebook 04's final cell, or upload `reduced_operator.npy` directly.

For unattended execution, set `CPS_EVIDENCE_PATH` to an evidence directory, a `cps-export.zip` archive, or a specific `reduced_operator.npy`.

In [ ]:
import os, pathlib, shutil, numpy as np

import_root = pathlib.Path("/content/cps-import")
import_root.mkdir(parents=True, exist_ok=True)

def reduced_operators_below(candidate):
    candidate = pathlib.Path(candidate).expanduser()
    if not candidate.exists():
        return []
    if candidate.is_file():
        if candidate.name == "reduced_operator.npy":
            return [candidate]
        if candidate.suffix.lower() == ".zip":
            destination = import_root / candidate.stem.replace(" ", "-")
            destination.mkdir(parents=True, exist_ok=True)
            shutil.unpack_archive(str(candidate), str(destination), "zip")
            return sorted(destination.rglob("reduced_operator.npy"))
        return []
    return sorted(candidate.rglob("reduced_operator.npy"))

evidence_path = os.environ.get("CPS_EVIDENCE_PATH")
search_locations = []
if evidence_path:
    explicit = pathlib.Path(evidence_path).expanduser()
    if not explicit.exists():
        raise FileNotFoundError(f"CPS_EVIDENCE_PATH does not exist: {explicit}")
    search_locations.append(explicit)
else:
    search_locations.extend([
        pathlib.Path("/content/cps-artifacts"),
        pathlib.Path("/content/cps-export"),
        pathlib.Path("/content/CPS/cps-artifacts"),
        pathlib.Path("/content/drive/MyDrive/cps-artifacts"),
        pathlib.Path("/content/drive/MyDrive/cps-export"),
    ])
    search_locations.extend(pathlib.Path("/content").glob("cps-export*.zip"))

paths = []
for location in search_locations:
    paths.extend(reduced_operators_below(location))

if not paths:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "No reduced operator was found. Set CPS_EVIDENCE_PATH to the prior "
            "evidence packet or copy it into /content/cps-artifacts."
        ) from exc

    print("[HANDOFF] This is a fresh Colab runtime.", flush=True)
    print("[HANDOFF] Upload cps-export.zip from notebook 04, or reduced_operator.npy.", flush=True)
    uploaded = files.upload()
    for uploaded_name, payload in uploaded.items():
        target = import_root / pathlib.Path(uploaded_name).name
        target.write_bytes(payload)
        paths.extend(reduced_operators_below(target))

if not paths:
    raise FileNotFoundError(
        "The uploaded material contains no reduced_operator.npy. Upload the "
        "cps-export.zip created by notebook 04's final cell."
    )

paths = sorted(set(pathlib.Path(item).resolve() for item in paths))
def packet_rank(operator_path):
    root = operator_path.parent
    companions = ["manifest.json", "basis.json", "couplings.json"]
    complete = sum((root / name).is_file() for name in companions)
    return complete, operator_path.stat().st_mtime, str(operator_path)

path = max(paths, key=packet_rank)
A = np.load(path)
print(f"[HANDOFF] candidates={len(paths)}; selected={path}", flush=True)
print(f"[PLAN] shape={A.shape}; ||A||₂={np.linalg.norm(A, 2):.6g}; ρ(A)={max(abs(np.linalg.eigvals(A))):.6g}", flush=True)


## Stage 2 — score every candidate, not only the winner

The table makes the decision legible. Risk combines phase-envelope spectral radius, finite-horizon gain, and inverse minimum gap. Lower is better under this preregistered surrogate.

In [ ]:
import pandas as pd
from IPython.display import display
from cps.controllers import score_candidate
from cps.pythia.planner import damping_family, plan_scalar_control

magnitude = np.abs(A).copy()
np.fill_diagonal(magnitude, 0.0)
flat = np.argsort(magnitude.ravel())[::-1]
edges=[]
for idx in flat:
    row,col=np.unravel_index(idx,magnitude.shape)
    if magnitude[row,col] <= 0 or len(edges) >= 8:
        break
    edges.append((int(row),int(col)))

candidates = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30]
rows=[]
for gamma in candidates:
    score=score_candidate(damping_family(A, gamma), edges)
    rows.append({
        "damping γ": gamma,
        "risk": score.risk,
        "spectral radius": score.spectral_radius,
        "transient gain": score.transient_gain,
        "minimum gap": score.minimum_gap,
    })
score_frame=pd.DataFrame(rows).sort_values("risk")
display(score_frame)
recommendation=plan_scalar_control(
    "isotropic_damping", 0.0, candidates,
    lambda gamma: damping_family(A, gamma), edges,
)
print("[PLAN] recommendation", recommendation.to_dict(), flush=True)

## Stage 3 — register the hypothesis

The next notebook must compare the recommended control against a baseline from identical weights and identical subsequent batches. Choosing the intervention after seeing continuation outcomes would invalidate the test.

In [ ]:
import json
plan_path = path.parent / "planner_recommendation.json"
plan_path.write_text(json.dumps(recommendation.to_dict(), indent=2))
print(f"[PLAN] wrote preregistered recommendation to {plan_path}", flush=True)

In [ ]:
from cps.notebook import export_artifacts
archive = export_artifacts(sources=(path.parent,))
print(f"Artifact archive ready for colab-cli download: {archive}", flush=True)